# Proyecto final Titanic - Resultado

**Estadistica Descriptiva e Inferencial**

Modelo de clasificacion inspirado en la Clase 12 de regresion logistica.

---

## Objetivo

Construir un modelo de clasificacion paso a paso para predecir si un pasajero sobrevivio o no sobrevivio en el Titanic.

En este resultado se usan solo cuatro variables predictoras:

```python
X = d[["sex", "Pclass", "age", "Fare"]]
y = d["Survived"]
```

Se eliminan las variables `familia` y `tiene_camarote` del proyecto original para medir de forma limpia que aporta cada una de estas cuatro variables: sexo, clase, edad y tarifa pagada.


---
## Celda 0 - Preparacion

Ejecuta esta celda tal cual. Carga los datos, prepara las variables y deja listo el dataset para el modelo.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings("ignore")

NAVY, BLUE, MAG, GREEN = "#0A2559", "#1A56E8", "#E6115E", "#12B886"
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

URL = ("https://raw.githubusercontent.com/josefrodrim/"
       "Estad-stica-Descriptiva-E-Inferencial/main/"
       "Proyecto_final_titanic/Data/Titanic-Dataset.csv")

try:
    titanic = pd.read_csv(URL)
    print("Datos cargados desde el repo del curso.")
except Exception:
    try:
        titanic = pd.read_csv("../Data/Titanic-Dataset.csv")
        print("Datos cargados desde ../Data/Titanic-Dataset.csv")
    except Exception:
        titanic = pd.read_csv("Titanic-Dataset.csv")
        print("Datos cargados desde archivo local en la carpeta del notebook.")

# Dataset de trabajo
d = titanic.copy()

# Variable numerica para sexo: 1 = mujer, 0 = hombre
d["sex"] = (d["Sex"] == "female").astype(int)

# Variable numerica para edad: se rellenan faltantes con la mediana
d["age"] = d["Age"].fillna(d["Age"].median())

# Alias para mantener compatibilidad con explicaciones/celdas posteriores
datos = d

# Variables finales del proyecto
X = d[["sex", "Pclass", "age", "Fare"]]
y = d["Survived"]

print(f"\n{len(datos)} pasajeros")
print(d[["Survived", "Sex", "sex", "Pclass", "age", "Fare"]].head())


### Variables del modelo

| Variable | Tipo | Que representa |
|---|---|---|
| `Survived` | objetivo | 1 si sobrevivio, 0 si no sobrevivio |
| `sex` | predictora | 1 si es mujer, 0 si es hombre |
| `Pclass` | predictora | clase del boleto: 1, 2 o 3 |
| `age` | predictora | edad, con faltantes rellenados con la mediana |
| `Fare` | predictora | tarifa pagada por el boleto |

La regresion logistica no predice directamente una etiqueta. Primero estima una probabilidad de sobrevivir. Despues convertimos esa probabilidad en `0` o `1` usando un umbral.


---
# Parte 1 - Reconocimiento

Antes de modelar, conviene mirar los datos. Esta parte sirve para formular hipotesis y detectar problemas de calidad.


### 1.1 - Valores faltantes


In [ ]:
faltantes = titanic.isna().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
print("Columnas con valores faltantes:")
print(faltantes)


### 1.2 - Tasas de supervivencia por grupo

Calculamos la supervivencia general y luego por las variables que usaremos en el modelo.


In [ ]:
print("Tasa de supervivencia general:")
print(f"{100 * datos['Survived'].mean():.2f} %")

print("\nTasa de supervivencia por sexo:")
print((datos.groupby("Sex")["Survived"].mean() * 100).round(2))

print("\nTasa de supervivencia por clase:")
print((datos.groupby("Pclass")["Survived"].mean() * 100).round(2))

print("\nResumen de Fare por resultado:")
print(datos.groupby("Survived")["Fare"].describe().round(2))


### 1.3 - Graficos exploratorios


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

(datos.groupby("Sex")["Survived"].mean() * 100).plot(
    kind="bar", ax=axes[0], color=[MAG, BLUE]
)
axes[0].set_title("Supervivencia por sexo", color=NAVY, fontweight="bold")
axes[0].set_ylabel("% sobrevivio")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)

(datos.groupby("Pclass")["Survived"].mean() * 100).plot(
    kind="bar", ax=axes[1], color=GREEN
)
axes[1].set_title("Supervivencia por clase", color=NAVY, fontweight="bold")
axes[1].set_ylabel("% sobrevivio")
axes[1].set_xlabel("Clase del boleto")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4.2))
for sobrevivio, color, label in [(0, MAG, "No sobrevivio"), (1, BLUE, "Sobrevivio")]:
    datos.loc[datos["Survived"] == sobrevivio, "Fare"].plot(
        kind="hist", bins=35, alpha=0.55, color=color, label=label
    )
plt.title("Distribucion de Fare por resultado", color=NAVY, fontweight="bold")
plt.xlabel("Fare")
plt.legend(frameon=False)
plt.show()


### 1.4 - Hipotesis antes de modelar

1. Las mujeres tendran mayor probabilidad de sobrevivir que los hombres.
2. Los pasajeros de primera clase tendran mayor probabilidad de sobrevivir que los de tercera clase.
3. La edad puede aportar informacion adicional: ninos y adultos pudieron tener probabilidades distintas de sobrevivir.
4. Los pasajeros que pagaron tarifas mas altas tenderan a tener mayor probabilidad de sobrevivir, aunque parte de ese efecto puede estar relacionado con la clase del boleto.


---
# Parte 2 - Modelos incrementales

Construimos modelos agregando una variable a la vez. Asi medimos que aporta cada variable.

| Modelo | Variables |
|---|---|
| M0 | ninguna: predecir siempre la clase mayoritaria |
| M1 | `sex` |
| M2 | `sex`, `Pclass` |
| M3 | `sex`, `Pclass`, `age` |
| M4 | `sex`, `Pclass`, `age`, `Fare` |

Las metricas se calculan siempre en datos de prueba.


### 2.1 - Separar entrenamiento y prueba


In [ ]:
X = d[["sex", "Pclass", "age", "Fare"]]
y = d["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"train = {len(X_train)} pasajeros")
print(f"test  = {len(X_test)} pasajeros")
print(f"supervivencia en train = {100 * y_train.mean():.2f} %")
print(f"supervivencia en test  = {100 * y_test.mean():.2f} %")


### 2.2 - Tabla de modelos

Para cada modelo calculamos:

- **AUC**: mide que tan bien ordena el modelo a sobrevivientes por encima de no sobrevivientes.
- **Exactitud**: porcentaje de aciertos usando umbral 0.5.


In [ ]:
modelos = {
    "M1": ["sex"],
    "M2": ["sex", "Pclass"],
    "M3": ["sex", "Pclass", "age"],
    "M4": ["sex", "Pclass", "age", "Fare"],
}

filas = []
modelos_entrenados = {}

# M0: regla trivial. Como la mayoria no sobrevivio, predice siempre 0.
pred_m0 = np.zeros(len(y_test), dtype=int)
filas.append({
    "modelo": "M0",
    "variables": "ninguna",
    "AUC": 0.5000,
    "exactitud": round(accuracy_score(y_test, pred_m0), 4),
})

for nombre, cols in modelos.items():
    modelo = LogisticRegression(max_iter=1000)
    modelo.fit(X_train[cols], y_train)
    modelos_entrenados[nombre] = modelo

    prob = modelo.predict_proba(X_test[cols])[:, 1]
    pred = (prob >= 0.5).astype(int)

    filas.append({
        "modelo": nombre,
        "variables": ", ".join(cols),
        "AUC": round(roc_auc_score(y_test, prob), 4),
        "exactitud": round(accuracy_score(y_test, pred), 4),
    })

tabla_modelos = pd.DataFrame(filas)
print(tabla_modelos.to_string(index=False))


### 2.3 - Lectura de la tabla

La comparacion correcta no es mirar solo el modelo final. Hay que mirar cuanto cambia el desempeno cuando agregamos cada variable.

- Si M1 mejora mucho frente a M0, entonces `sex` aporta bastante.
- Si M2 mejora frente a M1, entonces `Pclass` agrega informacion adicional al sexo.
- Si M3 mejora frente a M2, entonces `age` aporta informacion adicional despues de considerar sexo y clase.
- Si M4 mejora poco frente a M3, entonces `Fare` aporta poca informacion nueva despues de considerar sexo, clase y edad.

Esto ultimo puede pasar porque `Fare` y `Pclass` estan relacionadas: normalmente los boletos de primera clase cuestan mas que los de tercera clase.


### 2.4 - Coeficientes y odds ratio

Como en la Clase 12, los coeficientes de una regresion logistica estan en escala de log-odds. Para interpretarlos usamos:

```python
odds ratio = exp(coeficiente)
```


In [ ]:
filas_coef = []

for nombre, cols in modelos.items():
    modelo = modelos_entrenados[nombre]
    for variable, coef in zip(cols, modelo.coef_[0]):
        filas_coef.append({
            "modelo": nombre,
            "variable": variable,
            "coeficiente": round(coef, 4),
            "odds_ratio": round(np.exp(coef), 4),
        })

tabla_coef = pd.DataFrame(filas_coef)
print(tabla_coef.to_string(index=False))


### 2.5 - Hallazgo esperado

Un hallazgo probable es que `Fare` no aporte tanto cuando ya estan `Pclass` y `age` en el modelo.

Esto no significa que la tarifa no tenga relacion con la supervivencia. Significa que parte de la informacion de `Fare` ya estaba capturada por `Pclass`. En otras palabras: la tarifa y la clase del boleto miden aspectos parecidos del nivel socioeconomico del pasajero.


---
# Parte 3 - Matriz de confusion y umbral

Usamos el mejor modelo por AUC. En este proyecto normalmente sera M4, porque contiene las cuatro variables solicitadas.


In [ ]:
mejor_modelo = tabla_modelos.sort_values("AUC", ascending=False).iloc[0]["modelo"]
print(f"Mejor modelo por AUC: {mejor_modelo}")

cols_mejor = modelos[mejor_modelo]
modelo_final = modelos_entrenados[mejor_modelo]
prob_test = modelo_final.predict_proba(X_test[cols_mejor])[:, 1]
pred_05 = (prob_test >= 0.5).astype(int)

cm = confusion_matrix(y_test, pred_05)
vn, fp, fn, vp = cm.ravel()

exactitud = accuracy_score(y_test, pred_05)
precision = precision_score(y_test, pred_05, zero_division=0)
recall = recall_score(y_test, pred_05)
auc = roc_auc_score(y_test, prob_test)

print("Matriz de confusion con umbral 0.5")
print(f"{'':22}{'predijo MURIO':>16}{'predijo VIVIO':>16}")
print(f"{'realmente MURIO':22}{vn:>16}{fp:>16}")
print(f"{'realmente VIVIO':22}{fn:>16}{vp:>16}")
print()
print(f"AUC       = {auc:.4f}")
print(f"Exactitud = {exactitud:.4f}")
print(f"Precision = {precision:.4f}")
print(f"Recall    = {recall:.4f}")


### 3.1 - Probar varios umbrales

El umbral 0.5 no es obligatorio. Mover el umbral cambia el tipo de error que comete el modelo.


In [ ]:
filas_umbral = []

for umbral in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (prob_test >= umbral).astype(int)
    vn_, fp_, fn_, vp_ = confusion_matrix(y_test, pred).ravel()

    filas_umbral.append({
        "umbral": umbral,
        "VP": vp_,
        "FP": fp_,
        "FN": fn_,
        "VN": vn_,
        "exactitud": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, pred), 4),
    })

tabla_umbral = pd.DataFrame(filas_umbral)
print(tabla_umbral.to_string(index=False))


### 3.2 - Decision de umbral para la aseguradora

Escenario: una aseguradora debe provisionar dinero para indemnizaciones de pasajeros fallecidos.

| Error | Que pasa |
|---|---|
| Falso positivo | Predije que sobrevive, pero murio. No se provisiono dinero suficiente. |
| Falso negativo | Predije que muere, pero sobrevivio. Se provisiono dinero de mas. |

En este caso, el falso positivo parece mas grave porque implica pagar sin reserva. Por eso conviene no clasificar como sobreviviente demasiado facilmente. Una opcion razonable es usar un umbral mayor que 0.5, por ejemplo 0.6 o 0.7, dependiendo de cuanto quiera protegerse la aseguradora contra falta de provision.

La decision final no debe justificarse solo con la exactitud, sino con el costo relativo de cada tipo de error.


### 3.3 - Curva ROC


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, prob_test)

plt.plot(fpr, tpr, color=BLUE, lw=3, label=f"Modelo final AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], color=MAG, lw=2, ls="--", label="Azar AUC = 0.5")
plt.xlabel("Falsos positivos")
plt.ylabel("Verdaderos positivos / recall")
plt.title("Curva ROC", color=NAVY, fontweight="bold")
plt.legend(frameon=False, loc="lower right")
plt.show()


---
# Cierre

## Conclusiones

1. `sex` deberia ser la variable con mayor aporte, porque en Titanic las mujeres tuvieron una tasa de supervivencia mucho mayor que los hombres.
2. `Pclass` tambien deberia aportar, porque la supervivencia fue mayor en primera clase y menor en tercera clase.
3. `age` puede aportar informacion adicional porque la edad estuvo relacionada con prioridades de supervivencia.
4. `Fare` puede aportar, pero su aporte incremental puede ser menor porque esta relacionada con `Pclass`.
4. El mejor modelo no se debe evaluar solo con exactitud. Tambien importan AUC, precision, recall y la matriz de confusion.
5. El umbral es una decision de negocio: cambia cuantos falsos positivos y falsos negativos aceptamos.

## Limitaciones

1. El modelo usa pocas variables y deja fuera informacion potencialmente util como familia, puerto de embarque, titulo del nombre o camarote.
2. La relacion encontrada no debe leerse como causalidad. Por ejemplo, pagar mas tarifa no causa necesariamente sobrevivir; puede estar asociado a clase, ubicacion del camarote u otros factores.
3. El resultado esta basado en una muestra historica especifica. No necesariamente generaliza a otros barcos, epocas o condiciones.

## Presentacion sugerida en 5 slides

| # | Slide |
|---|---|
| 1 | Problema, datos y variables usadas |
| 2 | Reconocimiento: supervivencia por sexo, clase y tarifa |
| 3 | Tabla de modelos M0 a M4 |
| 4 | Hallazgo: aporte incremental de `age` y `Fare` despues de `Pclass` |
| 5 | Umbral elegido, errores y limitaciones |
